In [1]:
# # ── CELL 1: Install ──────────────────────────────────────────────────────────
# !pip install -q transformers datasets faiss-cpu camel-tools pyarabic
# !pip install -q torch tqdm pandas numpy
!pip install -q -U huggingface_hub sentence-transformers transformers datasets faiss-cpu camel-tools pyarabic
!pip install -q torch tqdm pandas numpy
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ── CELL 2: Load ARCD dataset ─────────────────────────────────────────────────
# ARCD = Arabic Reading Comprehension Dataset
# Format: context paragraph + question + answer
# We use it for search evaluation: question = query, context = relevant chunk

from datasets import load_dataset
import pandas as pd

dataset = load_dataset('hsseinmz/arcd', trust_remote_code=True)
print(dataset)
print('\nSample:')
print(dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hsseinmz/arcd' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/174k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/192k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/693 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/702 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 693
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 702
    })
})

Sample:
{'id': '969331847966', 'title': 'جمال خاشقجي', 'context': 'جمال أحمد حمزة خاشقجي (13 أكتوبر 1958، المدينة المنورة - 2 أكتوبر 2018)، صحفي وإعلامي سعودي، رأس عدّة مناصب لعدد من الصحف في السعودية، وتقلّد منصب مستشار، كما أنّه مدير عام قناة العرب الإخبارية سابقًا.', 'question': '- من هو جمال أحمد حمزة خاشقجي؟', 'answers': {'text': ['صحفي وإعلامي'], 'answer_start': [73]}}


In [3]:
# ── CELL 3: Prepare passages and queries ──────────────────────────────────────
import re

def normalize_arabic(text):
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    diacritics = re.compile(r'[\u064B-\u065F\u0670]')
    text = diacritics.sub('', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Build evaluation set
eval_data = []
seen_contexts = {}

for item in dataset['train']:
    ctx = normalize_arabic(item['context'])
    q   = normalize_arabic(item['question'])
    ans = normalize_arabic(item['answers']['text'][0] if item['answers']['text'] else '')

    # Use context as the 'passage' to index; question as the 'query'
    if ctx not in seen_contexts:
        seen_contexts[ctx] = len(seen_contexts)

    eval_data.append({
        'query': q,
        'relevant_passage_id': seen_contexts[ctx],
        'answer': ans,
    })

passages = list(seen_contexts.keys())
print(f'Passages to index: {len(passages)}')
print(f'Queries for eval:  {len(eval_data)}')

# Limit for speed
passages = passages[:500]
eval_data = [e for e in eval_data if e['relevant_passage_id'] < 500][:200]
print(f'Using {len(passages)} passages, {len(eval_data)} queries for eval')

Passages to index: 231
Queries for eval:  693
Using 231 passages, 200 queries for eval


In [4]:
# ── CELL 4: CAMeL-BERT Embedder ───────────────────────────────────────────────
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-msa'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
model.cuda()
print('Model loaded on GPU')

def mean_pool(token_emb, attention_mask):
    mask_exp = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return (token_emb * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, batch_size=64):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[i:i+batch_size]
        inp = tokenizer(batch, padding=True, truncation=True,
                        max_length=512, return_tensors='pt')
        inp = {k: v.cuda() for k, v in inp.items()}
        out = model(**inp)
        emb = mean_pool(out.last_hidden_state, inp['attention_mask'])
        emb = emb.cpu().numpy().astype(np.float32)
        # L2 normalize
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        emb = emb / np.where(norms == 0, 1, norms)
        all_embs.append(emb)
    return np.vstack(all_embs)

print('Encoding passages...')
passage_embeddings = encode(passages)
print(f'Passage embeddings shape: {passage_embeddings.shape}')

print('Encoding queries...')
queries = [e['query'] for e in eval_data]
query_embeddings = encode(queries)
print(f'Query embeddings shape: {query_embeddings.shape}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading CAMeL-Lab/bert-base-arabic-camelbert-msa...


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

...

Model loaded on GPU
Encoding passages...


Encoding: 100%|██████████| 4/4 [00:08<00:00,  2.09s/it]


Passage embeddings shape: (231, 768)
Encoding queries...


Encoding: 100%|██████████| 4/4 [00:00<00:00, 18.99it/s]

Query embeddings shape: (200, 768)


In [5]:
# ── CELL 5: Build FAISS index ─────────────────────────────────────────────────
import faiss

dim = passage_embeddings.shape[1]  # 768
index = faiss.IndexFlatIP(dim)     # Inner product = cosine on L2-normalized vecs

# Add directly to the CPU index
index.add(passage_embeddings)
print(f'FAISS index built: {index.ntotal} vectors')

FAISS index built: 231 vectors


In [6]:
# ── CELL 6: Precision@K and Recall@K evaluation ───────────────────────────────
import pandas as pd

def precision_at_k(retrieved_ids, relevant_id, k):
    top_k = retrieved_ids[:k]
    return 1.0 if relevant_id in top_k else 0.0

def recall_at_k(retrieved_ids, relevant_id, k):
    # For single-relevant case: recall = 1 if found in top-k, else 0
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0

def evaluate_retrieval(query_embeddings, relevant_ids, index, k_values=[1,3,5,10]):
    scores = {f'P@{k}': [] for k in k_values}
    scores.update({f'R@{k}': [] for k in k_values})

    max_k = max(k_values)
    _, all_indices = index.search(query_embeddings, max_k)

    for i, relevant_id in enumerate(relevant_ids):
        retrieved = all_indices[i].tolist()
        for k in k_values:
            scores[f'P@{k}'].append(precision_at_k(retrieved, relevant_id, k))
            scores[f'R@{k}'].append(recall_at_k(retrieved, relevant_id, k))

    return {metric: round(np.mean(vals), 4) for metric, vals in scores.items()}

relevant_ids = [e['relevant_passage_id'] for e in eval_data]
results = evaluate_retrieval(query_embeddings, relevant_ids, index)

print('=== Search Evaluation Results ===')
print('Metric | Score')
print('-------|------')
for metric, score in results.items():
    print(f'{metric:6s} | {score:.4f}')

=== Search Evaluation Results ===
Metric | Score
-------|------
P@1    | 0.4000
P@3    | 0.6050
P@5    | 0.7300
P@10   | 0.8250
R@1    | 0.4000
R@3    | 0.6050
R@5    | 0.7300
R@10   | 0.8250


In [7]:
# ── CELL 7: Chunking Ablation + Re-Ranking ───────────────────────────────────
# Simulate different chunk sizes and test FAISS vs FAISS+Cross-Encoder

import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import CrossEncoder

def split_into_chunks(text, max_words):
    words = text.split()
    return [' '.join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

print("Loading Multilingual Cross-Encoder...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=device)

ablation_configs = [
    {'name': 'Small (50 words)',  'max_words': 50},
    {'name': 'Medium (100 words)', 'max_words': 100},
    {'name': 'Large (200 words)', 'max_words': 200},
]

ablation_results = []

# Keep the original passage IDs so eval queries can map correctly
long_passages_with_ids = [(i, p) for i, p in enumerate(passages) if len(p.split()) > 100][:100]

for cfg in ablation_configs:
    print(f"\n{'='*40}\nConfig: {cfg['name']}\n{'='*40}")
    all_chunks = []
    chunk_to_passage = []  # maps chunk_idx -> ORIGINAL passage_idx

    for orig_idx, passage in long_passages_with_ids:
        chunks = split_into_chunks(passage, cfg['max_words'])
        for chunk in chunks:
            all_chunks.append(chunk)
            chunk_to_passage.append(orig_idx)

    print(f'Total chunks: {len(all_chunks)} (avg per passage: {len(all_chunks)/len(long_passages_with_ids):.1f})')

    # Embed chunks for FAISS
    chunk_embs = encode(all_chunks, batch_size=64)

    # Build mini index directly on CPU/GPU depending on your setup
    chunk_index = faiss.IndexFlatIP(dim)
    chunk_index.add(chunk_embs)

    # Only use evaluation queries that target the long passages we just indexed
    valid_long_ids = set([i for i, p in long_passages_with_ids])
    ablation_queries = [e for e in eval_data if e['relevant_passage_id'] in valid_long_ids][:50]
    
    if not ablation_queries:
        print("Warning: No evaluation queries target these specific long passages.")
        continue

    ablation_q_embs = encode([e['query'] for e in ablation_queries], batch_size=64)
    ablation_relevant = [e['relevant_passage_id'] for e in ablation_queries]

    k_values = [1, 3, 5]
    k_retrieval = 10 # We retrieve top 10 with FAISS, then re-rank them
    
    # Search the index
    _, indices = chunk_index.search(ablation_q_embs, k_retrieval)

    # Dictionaries to hold our two sets of scores
    baseline_scores = {f'P@{k}': [] for k in k_values}
    reranked_scores = {f'P@{k}': [] for k in k_values}

    print("Evaluating FAISS Baseline and Running Cross-Encoder...")
    for i, rel_passage_id in enumerate(ablation_relevant):
        query_text = ablation_queries[i]['query']
        retrieved_chunk_ids = indices[i].tolist()
        retrieved_chunk_ids = [c for c in retrieved_chunk_ids if c >= 0]
        
        # 1. FAISS BASELINE EVALUATION
        retrieved_passage_ids_faiss = [chunk_to_passage[c] for c in retrieved_chunk_ids]
        for k in k_values:
            hit = 1.0 if rel_passage_id in retrieved_passage_ids_faiss[:k] else 0.0
            baseline_scores[f'P@{k}'].append(hit)
            
        # 2. CROSS-ENCODER RE-RANKING
        retrieved_texts = [all_chunks[c] for c in retrieved_chunk_ids]
        cross_inp = [[query_text, text] for text in retrieved_texts]
        
        if cross_inp: # Ensure we actually retrieved something
            cross_scores = reranker.predict(cross_inp)
            
            # Pair scores with their parent passage IDs and sort
            paired_results = list(zip(cross_scores, retrieved_passage_ids_faiss))
            paired_results.sort(key=lambda x: x[0], reverse=True)
            reranked_passage_ids = [passage_id for score, passage_id in paired_results]
            
            # Evaluate Re-ranked results
            for k in k_values:
                hit = 1.0 if rel_passage_id in reranked_passage_ids[:k] else 0.0
                reranked_scores[f'P@{k}'].append(hit)
        else:
            for k in k_values:
                reranked_scores[f'P@{k}'].append(0.0)

    # Calculate averages and append to our results list
    avg_base = {m: round(np.mean(v), 4) for m, v in baseline_scores.items()}
    avg_rerank = {m: round(np.mean(v), 4) for m, v in reranked_scores.items()}
    
    ablation_results.append({'Config': f"{cfg['name']} (FAISS Only)", **avg_base})
    ablation_results.append({'Config': f"{cfg['name']} (+ Re-Rank)", **avg_rerank})
    
    print(f"  FAISS Only: {avg_base}")
    print(f"  + Re-Rank : {avg_rerank}")

print('\n=== Final Chunking & Re-Ranking Results ===')
if ablation_results:
    df = pd.DataFrame(ablation_results)
    print(df.to_string(index=False))

    # Save results
    df.to_csv('/kaggle/working/chunking_reranking_results.csv', index=False)
    print('\nSaved to chunking_reranking_results.csv')

2026-05-03 01:29:21.533964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777771761.738706     130 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777771761.794621     130 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777771762.278478     130 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777771762.278525     130 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777771762.278528     130 computation_placer.cc:177] computation placer alr

Loading Multilingual Cross-Encoder...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Evaluating FAISS Baseline and Running Cross-Encoder...
  FAISS Only: {'P@1': 0.64, 'P@3': 0.8, 'P@5': 0.88}
  + Re-Rank : {'P@1': 0.86, 'P@3': 0.9, 'P@5': 0.9}

Config: Medium (100 words)
Total chunks: 166 (avg per passage: 2.2)


Encoding: 100%|██████████| 1/1 [00:00<00:00, 18.95it/s]


Evaluating FAISS Baseline and Running Cross-Encoder...
  FAISS Only: {'P@1': 0.58, 'P@3': 0.76, 'P@5': 0.88}
  + Re-Rank : {'P@1': 0.84, 'P@3': 0.9, 'P@5': 0.9}

Config: Large (200 words)
Total chunks: 89 (avg per passage: 1.2)


Encoding: 100%|██████████| 1/1 [00:00<00:00, 17.80it/s]


Evaluating FAISS Baseline and Running Cross-Encoder...
  FAISS Only: {'P@1': 0.6, 'P@3': 0.78, 'P@5': 0.86}
  + Re-Rank : {'P@1': 0.82, 'P@3': 0.92, 'P@5': 0.92}

=== Final Chunking & Re-Ranking Results ===
                         Config  P@1  P@3  P@5
  Small (50 words) (FAISS Only) 0.64 0.80 0.88
   Small (50 words) (+ Re-Rank) 0.86 0.90 0.90
Medium (100 words) (FAISS Only) 0.58 0.76 0.88
 Medium (100 words) (+ Re-Rank) 0.84 0.90 0.90
 Large (200 words) (FAISS Only) 0.60 0.78 0.86
  Large (200 words) (+ Re-Rank) 0.82 0.92 0.92

Saved to chunking_reranking_results.csv


In [8]:
# ── CELL 8: Save embeddings for local use (FIXED) ────────────────────────────
import json
import numpy as np
import faiss

print("Rebuilding the optimized 50-word chunk database for saving...")

# 1. Rebuild the 50-word chunks specifically
all_chunks_50 = []
chunk_to_passage_50 = []

# Uses the long_passages_with_ids from Cell 7
for orig_idx, passage in long_passages_with_ids:
    chunks = split_into_chunks(passage, 50) # Force the 50-word config
    for chunk in chunks:
        all_chunks_50.append(chunk)
        chunk_to_passage_50.append(orig_idx)

print("Encoding 50-word chunks...")
chunk_embs_50 = encode(all_chunks_50, batch_size=64)

# Build the FAISS index
chunk_index_50 = faiss.IndexFlatIP(dim)
chunk_index_50.add(chunk_embs_50)

print("Saving files to Kaggle/working...")

# 2. Save the embeddings
np.save('/kaggle/working/chunk_embeddings_50.npy', chunk_embs_50)

# 3. Save the FAISS index
faiss.write_index(chunk_index_50, '/kaggle/working/arcd_chunk_index_50.faiss')

# 4. Save the chunks + metadata as JSON
with open('/kaggle/working/chunks_50.json', 'w', encoding='utf-8') as f:
    json.dump([
        {
            'chunk_id': i, 
            'original_passage_id': int(chunk_to_passage_50[i]), # Cast to int to prevent errors
            'text': chunk_text
        } for i, chunk_text in enumerate(all_chunks_50)
    ], f, ensure_ascii=False, indent=2)

print('Saved: chunk_embeddings_50.npy, arcd_chunk_index_50.faiss, chunks_50.json')
print('Download these from Kaggle Output → put in data/index/ locally')

Rebuilding the optimized 50-word chunk database for saving...
Encoding 50-word chunks...


Encoding: 100%|██████████| 5/5 [00:01<00:00,  3.10it/s]

Saving files to Kaggle/working...
Saved: chunk_embeddings_50.npy, arcd_chunk_index_50.faiss, chunks_50.json
Download these from Kaggle Output → put in data/index/ locally


In [9]:
# ── CELL 9: Generative QA (RAG) ──────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

print("1. Loading the Generative LLM (Qwen 2.5 - 1.5B)...")
llm_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# CRITICAL FIX: We name these 'llm_tokenizer' and 'llm_model' 
# so we don't overwrite your CAMeL-BERT 'tokenizer' and 'model' from Cell 4!
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch.float16, 
    device_map="auto" # Automatically maps to your Kaggle GPU
)

rag_pipeline = pipeline(
    "text-generation", 
    model=llm_model, 
    tokenizer=llm_tokenizer,
    max_new_tokens=150
)

print("\n2. Testing the Full RAG Pipeline on a sample query...")

# Let's take the first query from our evaluation set (or type your own!)
sample_query_idx = 0
user_question = ablation_queries[sample_query_idx]['query']

# --- THE RAG PIPELINE (Inference) ---

# Step A: Embed the question and search FAISS (Top 10)
# This safely uses the encode() function and CAMeL-BERT model from Cell 4
user_q_emb = encode([user_question]) 
_, faiss_indices = chunk_index_50.search(user_q_emb, 10)

# Filter out empty indices (-1) if FAISS didn't find 10 results
retrieved_chunk_ids = [c for c in faiss_indices[0].tolist() if c >= 0]
retrieved_texts = [all_chunks_50[c] for c in retrieved_chunk_ids]

# Step B: Re-Rank with Cross-Encoder
cross_inp = [[user_question, text] for text in retrieved_texts]
cross_scores = reranker.predict(cross_inp)

# Pair the actual texts with their new scores and sort them highest to lowest
scored_texts = list(zip(cross_scores, retrieved_texts))
scored_texts.sort(key=lambda x: x[0], reverse=True)

# Grab the absolute best text
best_chunk_text = scored_texts[0][1]

# Step C: Build the strict RAG Prompt in Arabic
prompt = f"""أجب على السؤال التالي بناءً على السياق المقدم فقط. إذا لم تكن الإجابة موجودة في السياق، قل "لا أعرف".

السياق:
{best_chunk_text}

السؤال: {user_question}
الإجابة:"""

# Format the prompt using Qwen's chat template
messages = [
    {"role": "system", "content": "أنت مساعد ذكي ومفيد يجيب على الأسئلة بدقة بناءً على النصوص المقدمة فقط."},
    {"role": "user", "content": prompt}
]
formatted_prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Step D: Generate the answer!
print("Generating answer...\n")
outputs = rag_pipeline(formatted_prompt, do_sample=False)
generated_text = outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1].strip()

print("="*60)
print(f"QUESTION : {user_question}")
print("-" * 60)
print(f"RETRIEVED (Top #1): {best_chunk_text}")
print("-" * 60)
print(f"ANSWER   : {generated_text}")
print("="*60)

1. Loading the Generative LLM (Qwen 2.5 - 1.5B)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


2. Testing the Full RAG Pipeline on a sample query...


Encoding: 100%|██████████| 1/1 [00:00<00:00, 46.56it/s]

Generating answer...




/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


QUESTION : - ما هي الكيانات الاولي في المملكه العربيه السعوديه؟
------------------------------------------------------------
RETRIEVED (Top #1): حكم ال سعود تاريخيا في نجد ومناطق واسعه من الجزيره العربيه اكثر من مره، وتعتبر المملكه السعوديه الحاليه نتاجا ووارثه لتلك الكيانات التاريخيه، اول تلك الكيانات اماره الدرعيه التي اسسها محمد بن سعود سنه 1157 ه / 1744 وظلت حتي قاد ابراهيم باشا جيش والي مصر العثماني في حمله للقضاء
------------------------------------------------------------
ANSWER   : الكيانات الأولى في المملكة العربية السعودية كانت امبراطورية الدرعية التي أسسها محمد بن سعود سنة 1157 هـ / 1744م.
